# 06 — Integración de voz y compra

Conectamos audio en español, transcripción revisable, agente con RAG y herramientas, tienda SQLite y respuesta generada con XTTS-v2. Qwen se mantiene en GPU; Whisper, recuperación y síntesis usan CPU en la integración para dejar margen de VRAM.


In [ ]:
%pip -q install -r requirements-lab4-modular.txt

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
from pathlib import Path
from IPython.display import Audio, display
from lab4_asistente import config as C
from lab4_asistente.runtime import cargar_sistema, cargar_modelos_voz


## Seleccionar el perfil de voz

El menú muestra los perfiles disponibles y conserva la identidad seleccionada para el turno de prueba.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

VOICES = C.voces_disponibles()
if not VOICES:
    raise FileNotFoundError("No se encontraron grabaciones de voz.")
voice_selector = widgets.Dropdown(options=sorted(VOICES), description="Perfil de voz:")
display(voice_selector)
print("Elige la voz y ejecuta la siguiente celda para cargar la integración.")


## Cargar la solución integrada

La GPU queda reservada para Qwen. Los modelos de recuperación, reconocimiento y síntesis se ubican en CPU para limitar la ocupación simultánea de VRAM.


In [ ]:
COMP = cargar_sistema(cargar_llm=True)
SIS = COMP["sistema"]
VOICE_ID = voice_selector.value
ASR, TTS, cargas_voz = cargar_modelos_voz(VOICE_ID)
CARGAS_INICIALES = {**COMP["cargas"], **cargas_voz}
print("Voz seleccionada:", VOICE_ID)
print("Dispositivo LLM:", COMP["dispositivos"]["llm"])
print("Dispositivos ASR/TTS:", cargas_voz["dispositivo_asr"], "/", cargas_voz["dispositivo_tts"])
print("Cargas iniciales (fuera de la latencia por turno):", CARGAS_INICIALES)


## Procesar una consulta hablada y permitir corregirla

In [ ]:
from pathlib import Path

archivo = input("Ruta del WAV del usuario (ej. data/voice_samples/consulta.wav): ").strip()
ruta_audio = Path(archivo).expanduser()
if not ruta_audio.is_absolute():
    ruta_audio = ROOT / ruta_audio
if not ruta_audio.exists():
    raise FileNotFoundError(f"No se encontró el audio: {ruta_audio}")
transcripcion = ASR.transcribir(ruta_audio)
print("Transcripción ASR:", transcripcion)
corregida = input("Corrige la transcripción si hace falta; Enter conserva la original: ").strip()
entrada = corregida or transcripcion
respuesta = SIS.agente.turno(entrada)
print("Respuesta:", respuesta["respuesta"])
print("Operaciones:", [(o["herramienta"], o["resultado"].get("ok")) for o in respuesta["operaciones"]])
ruta_salida = C.RUTA_OUTPUT / "respuesta_demo.wav"
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
TTS.sintetizar(respuesta["respuesta"], ruta_salida)
display(Audio(filename=str(ruta_salida)))


## Qué comprobar en la demostración

La transcripción se revisa antes de llegar al agente. El agente solo accede al catálogo sin etiquetas, consulta precios/stock por herramientas y solicita confirmación después de mostrar la cotización. XTTS usa el perfil `data/voice_profiles/<VOICE_ID>.pt`; la grabación de referencia se resuelve desde el catálogo de voces del repositorio.
